In [1]:
import os

import matplotlib.pyplot as plt

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
import scipy.signal
from typing import List, Callable, Iterable, Tuple
from tqdm import tqdm 
import functools

from msmjax.gridops_multidim import BSplineInterpolationAxis, BSplineInterpolationGrid
from msmjax.gridops_multidim import set_up_grid_axis
from msmjax.gridops_multidim import set_up_grids_all_levels

from msmjax.gridops_multidim import multi_inds_from_individual_axes_inds, \
    arbitrary_dim_outer
from msmjax.gridops_multidim import create_anterpolation_operator
from msmjax.gridops_multidim import create_restriction_operator_1d, \
    create_restriction_operator
from msmjax.gridops_multidim import create_prolongation_operator_1d, create_prolongation_operator
from msmjax.gridops_multidim import create_interaction_operator, create_custom_interaction_operator_2
from msmjax.gridops_multidim import create_all_grid_to_grid_ops
from msmjax.gridops_multidim import create_compute_U_oneplus, \
    create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus


In [2]:
%matplotlib notebook

# Helper functions

In [3]:
# see https://gitlab.tuwien.ac.at/e165-03-1_theoretische_materialchemie/scripts-et-al/-/wikis/Sqrt-without-trivial-NaN-derivative-for-jax

@jax.custom_jvp
def _sqrt(x):
    return jnp.sqrt(x)

@_sqrt.defjvp
def _sqrt_jvp(primals, tangents):
    x, = primals
    xdot, = tangents
    primal_out = _sqrt(x)
    tangent_out = jnp.where(x == 0., 0., 0.5 / primal_out) * xdot
    return (primal_out, tangent_out)

# Basic settings

In [4]:
# geometry
length = 10.0
level_one_gridspacing = 1.25
ndim = 3

# MSM
max_gridlevel = 4

# splines
p = 6
order = p - 1

# particles
n_particles = 15

In [5]:
J_zeroplus = compute_J_zeroplus(p)
J = jnp.concatenate((J_zeroplus[::-1][:-1], J_zeroplus))

## Create particle configuration

In [6]:
# TODO: change back to more particles and random charges
#  (few particles and hard-coded charges serve visualization purposes only)

rng = onp.random.default_rng(1632794)
pos = rng.uniform(0., length, size=(n_particles, ndim))
chg = rng.uniform(-1., 1., size=n_particles)
# chg = onp.array([-2, -2, 1, 1, 1])

# Non-periodic boundaries

## Construct grids


In [7]:
grids_all_levels = set_up_grids_all_levels(
    box_lengths=[length] * ndim,
    level_one_spacings=[level_one_gridspacing] * ndim,
    pbcs=[False] * ndim,
    n_levels=max_gridlevel,
    p=p,
    J_zeroplus=J_zeroplus,
)

grid_level_one = grids_all_levels[1]
grid_level_two = grids_all_levels[2]

## Construct kernel stencils

In [8]:
sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmjax.kernels import SofteningFunctionOneOverR, split_one_over_r_kernel
from msmfornn.gridtools import construct_grids_all_levels
from msmfornn.splines.coefficients import compute_coeffs_withtruncation
from msmfornn.grid_to_grid_mapping import \
    compute_kernel_stencils_all_gridlevels

In [9]:
# We're using equal grid spacings in all directions here
level_one_gridspacing = grids_all_levels[1].axes[0].h
alpha = 3
level_zero_cutoff = alpha * level_one_gridspacing

params_oldmsm = {
    "min_pos": onp.array([0.] * grid_level_one.ndim),
    "max_pos": onp.array([a.length for a in grid_level_one.axes]),
    "level_one_gridspacing": level_one_gridspacing,
    "level_zero_cutoff": level_zero_cutoff,
    "max_gridlevel": max_gridlevel,
    "p": p,
    "mu": 3
}

grids_oldmsm = construct_grids_all_levels(
    min_pos=params_oldmsm["min_pos"],
    max_pos=params_oldmsm["max_pos"],
    p=params_oldmsm["p"],
    level_one_gridspacing=params_oldmsm["level_one_gridspacing"],
    max_gridlevel=params_oldmsm["max_gridlevel"],
)
partial_kernels = split_one_over_r_kernel(
    max_level=max_gridlevel,
    level_zero_cutoff=level_zero_cutoff,
    softening_function=SofteningFunctionOneOverR(p),
)
omega, _ = compute_coeffs_withtruncation(p=params_oldmsm["p"],
                                         mu=params_oldmsm["mu"])
omega_zeroplus = omega[len(omega) // 2:]

kernelstencils_old = compute_kernel_stencils_all_gridlevels(
    kernelfunctions=partial_kernels,
    grids=grids_oldmsm,
    level_zero_cutoff=level_zero_cutoff,
    omega_zeroplus=omega_zeroplus,
)
kernelstencils_old_symmetric = [None]
for stncl in kernelstencils_old[1:]:
    pw = [(s - 1, 0) for s in stncl.shape]
    stncl_symm = jnp.pad(stncl, pad_width=pw, mode='reflect')
    kernelstencils_old_symmetric.append(stncl_symm)

## Test (some of) the individual operations

### Anterpolation

In [10]:
@jax.jit
def evaluate_bspline_basis_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_multiparticle(pos)


@jax.jit
def evaluate_bspline_basis_gradient_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_gradient_multiparticle(pos)

In [11]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_multiparticle(pos)

211 µs ± 39.7 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_gradient_multiparticle(pos)

326 µs ± 78.7 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
vals, inds = evaluate_bspline_basis_multiparticle(pos)
grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)

In [14]:
@jax.jit
def combined(pos):
    vals, inds = evaluate_bspline_basis_multiparticle(pos)
    grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)
    return inds, vals, grads

In [15]:
jax.device_put(pos)
%timeit combined(pos)

403 µs ± 52.8 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
anterpolate_level_one = create_anterpolation_operator(grid=grid_level_one)
jitted_anterpolate_level_one = jax.jit(anterpolate_level_one)

In [17]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate_level_one(pos, chg).block_until_ready()

538 µs ± 174 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
gridcharge_level_one = anterpolate_level_one(pos, chg)

#### Test grid charges summing to net particle charge

In [19]:
anterpolation_funcs_all_levels = [None] + [
    jax.jit(create_anterpolation_operator(g)) for g in grids_all_levels[1:]]

gridcharges_direct = [None] + [af(pos, chg) for af in
                               anterpolation_funcs_all_levels[1:]]


In [20]:
net_charge = chg.sum()

for gc in gridcharges_direct[1:]:
    assert jnp.isclose(gc.sum(), net_charge)

#### Plot particles and grid charge

In [21]:
flat_indices = jnp.arange(gridcharge_level_one.size)
unraveled_indices = jnp.unravel_index(flat_indices, grid_level_one.shape)
grid_points_individual_axes = []
for i, axis in enumerate(grid_level_one.axes):
    points = axis.to_raw_indices(unraveled_indices[i]) * axis.h
    grid_points_individual_axes.append(points)

In [22]:
try:
    mask = onp.abs(gridcharge_level_one.ravel()) >= 0.01
    
    x, y, z, = grid_points_individual_axes
    
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    ax.scatter(x.ravel()[mask], y.ravel()[mask], z.ravel()[mask], c=gridcharge_level_one.ravel()[mask], s=10)
    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=chg, s=100)
    
    plt.show()
except ValueError:
    pass

<IPython.core.display.Javascript object>

### Restriction

In [23]:
restriction_funcs_all_levels = [None, None]
for lvl in range(2, len(grids_all_levels)):
    rf = create_restriction_operator(grid_source_fine=grids_all_levels[lvl - 1],
                                     grid_target_coarse=grids_all_levels[lvl])
    rf = jax.jit(rf)
    restriction_funcs_all_levels.append(rf)

# TODO: make output shapes of calculation via interpolation and restriction compatible
#  (the former currently returns a flat array, the latter a multi-dimensional one)
gridcharges_via_restriction = [None, gridcharges_direct[1].reshape(grids_all_levels[1].shape)]
for lvl in range(2, len(grids_all_levels)):
    rf = restriction_funcs_all_levels[lvl]
    gc_lowergrid = gridcharges_via_restriction[lvl - 1]
    gc = rf(gc_lowergrid)
    gridcharges_via_restriction.append(gc)

In [24]:
for gc_direct, gc_restrict in zip(gridcharges_direct[2:],
                                  gridcharges_via_restriction[2:]):
    assert jnp.allclose(gc_direct, gc_restrict) # TODO: shapes

### Prolongation

In [25]:
prolongate = create_prolongation_operator(
    grid_target_fine=grids_all_levels[1],
    grid_source_coarse=grids_all_levels[2],
)

In [26]:
gridcharges_via_restriction[2].shape

(11, 11, 11)

In [27]:
prolongate(gridcharges_via_restriction[2]).shape

(15, 15, 15)

### Interaction

In [28]:
kernelstencil_level_one = kernelstencils_old_symmetric[1]
gridcharge_in = gridcharge_level_one.reshape(grid_level_one.shape)

@jax.jit
def interact_custom(in_array):
    return create_interaction_operator(grid_level_one, kernelstencil_level_one)(in_array)

@jax.jit
def interact_custom_2(in_array):
    return create_custom_interaction_operator_2(grid_level_one, kernelstencil_level_one)(in_array)

@jax.jit
def interact_convolve_direct_jax(in_array):
    return jax.scipy.signal.convolve(in_array, kernelstencil_level_one, mode="same", method="direct")

@jax.jit
def interact_convolve_fft_jax(in_array):
    return jax.scipy.signal.convolve(in_array, kernelstencil_level_one, mode="same", method="fft")

def interact_convolve_direct_scipy(in_array):
    return scipy.signal.convolve(in_array, onp.array(kernelstencil_level_one), mode="same", method="direct")

def interact_convolve_fft_scipy(in_array):
    return scipy.signal.convolve(in_array, onp.array(kernelstencil_level_one), mode="same", method="fft")


In [29]:
result_custom = interact_custom(gridcharge_in)
result_custom_2 = interact_custom_2(gridcharge_in)
result_builtin_conv_jax = interact_convolve_direct_jax(gridcharge_in)
result_builtin_conv_scipy = interact_convolve_direct_scipy(gridcharge_in)

assert jnp.allclose(result_custom, result_custom_2)
assert jnp.allclose(result_custom, result_builtin_conv_jax)
assert jnp.allclose(result_custom, result_builtin_conv_scipy)

In [30]:
jax.device_put(gridcharge_in)

print("custom:")
%timeit interact_custom(gridcharge_in).block_until_ready()
print()

print("custom_2:")
%timeit interact_custom_2(gridcharge_in).block_until_ready()
print()
print()

print("jax.scipy.signal.convolve(..., method='direct'):")
%timeit interact_convolve_direct_jax(gridcharge_in).block_until_ready()
print()

print("jax.scipy.signal.convolve(..., method='fft'):")
%timeit interact_convolve_fft_jax(gridcharge_in).block_until_ready()
print()
print()

arr = onp.array(gridcharge_in)

print("scipy.signal.convolve(..., method='direct'):")
%timeit interact_convolve_direct_scipy(arr)
print()

print("scipy.signal.convolve(..., method='fft'):")
%timeit interact_convolve_fft_scipy(arr)
print()
print()

custom:
2.19 ms ± 54.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)

custom_2:
888 µs ± 13.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


jax.scipy.signal.convolve(..., method='direct'):
1.24 ms ± 9.28 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

jax.scipy.signal.convolve(..., method='fft'):
152 µs ± 1.01 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


scipy.signal.convolve(..., method='direct'):
118 ms ± 504 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)

scipy.signal.convolve(..., method='fft'):
727 µs ± 104 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Compare different convolution methods

In [31]:
restrict_fns, prolong_fns, interact_fns_custom = create_all_grid_to_grid_ops(
    grids=grids_all_levels,
    kernel_stencils=kernelstencils_old_symmetric,
)
restrict_fns, prolong_fns, interact_fns_direct = create_all_grid_to_grid_ops(
    grids=grids_all_levels,
    kernel_stencils=kernelstencils_old_symmetric,
    convolution_methods=[None] + ["scipy-direct"] * (len(grids_all_levels) - 1),
)
restrict_fns, prolong_fns, interact_fns_fft = create_all_grid_to_grid_ops(
    grids=grids_all_levels,
    kernel_stencils=kernelstencils_old_symmetric,
    convolution_methods=[None] + ["scipy-fft"] * (len(grids_all_levels) - 1),
)

In [32]:
out_custom = interact_fns_custom[1](gridcharge_in)

In [33]:
out_direct = interact_fns_direct[1](gridcharge_in)

In [34]:
out_fft = interact_fns_fft[1](gridcharge_in)

In [35]:
jnp.allclose(out_custom, out_direct)

Array(True, dtype=bool)

In [36]:
jnp.allclose(out_direct, out_fft)

Array(True, dtype=bool)

In [37]:
jnp.allclose(out_custom, result_custom)

Array(True, dtype=bool)

In [38]:
jnp.allclose(interact_fns_fft[1].keywords["in2"], kernelstencils_old_symmetric[1])

Array(True, dtype=bool)

In [39]:
jnp.allclose(interact_fns_direct[3].keywords["in2"], kernelstencils_old_symmetric[3])

Array(True, dtype=bool)

## Helper functions

In [40]:
@jax.jit
def kernelsum_1_to_L(r):
    return jnp.sum(jnp.asarray([k(r) for k in partial_kernels[1:]]))


def compute_U_oneplus_exact_loop(positions, charges):
    U_oneplus_direct = 0.
    for i in tqdm(range(len(positions))):
        for j in range(0, len(positions)):
            r_ij = jnp.linalg.norm(positions[i] - positions[j])
            U_oneplus_direct += charges[i] * charges[
                j] * kernelsum_1_to_L(r_ij)
    U_oneplus_direct *= 0.5

    return U_oneplus_direct

@jax.jit
def compute_U_oneplus_exact_vectorized(positions, charges):
    r_ij_vectors = positions[:, jnp.newaxis, :] - positions
    # To avoid NaN gradients, we need to use a custom
    # square root function here, since the distances can be zero 
    r_ij = _sqrt((r_ij_vectors * r_ij_vectors).sum(axis=2))
    K_ij = jax.vmap(jax.vmap(kernelsum_1_to_L))(r_ij)
    qi_qj = charges[:, jnp.newaxis] * charges
    
    return 0.5 * (qi_qj * K_ij).sum()

@jax.jit
def compute_U_oneplus_grid(positions, charges):
    return create_compute_U_oneplus(
        grids=grids_all_levels,
        kernel_stencils=kernelstencils_old_symmetric,
    )(positions, charges)

In [41]:
k_prime = jax.jit(jax.grad(kernelsum_1_to_L))


def compute_f_oneplus_exact_loop(positions, charges):
    n_dim = positions.shape[1]
    f_oneplus_direct = []
    for i in tqdm(range(len(positions))):
        force = jnp.zeros(n_dim)
        for j in range(len(positions)):
            if i == j:
                continue
            r_ij_vector = positions[i] - positions[j]
            r_ij = jnp.linalg.norm(r_ij_vector)
            force -= charges[j] * (r_ij_vector / r_ij) * k_prime(r_ij)
        force *= charges[i]
        f_oneplus_direct.append(force)

    return jnp.array(f_oneplus_direct)


@jax.jit
def compute_f_oneplus_exact_vectorized(positions, charges):
    return -jax.grad(compute_U_oneplus_exact_vectorized, argnums=0)(
        positions, charges
    )


@jax.jit
def compute_U_and_f_oneplus_grid(positions, charges):
    return create_compute_U_and_f_oneplus(
        grids=grids_all_levels,
        kernel_stencils=kernelstencils_old_symmetric,
    )(positions, charges)

## Test end-to-end

### Energy

In [42]:
U_oneplus_reference = compute_U_oneplus_exact_vectorized(pos, chg)
U_oneplus_grid = compute_U_oneplus_grid(pos, chg)
print("exact (loop)      :", U_oneplus_reference)
print("approx. (grid)    :", U_oneplus_grid)

assert jnp.isclose(U_oneplus_reference, compute_U_oneplus_exact_loop(pos, chg))

exact (loop)      : 3.4844547201472587
approx. (grid)    : 3.46141002945922


100%|██████████| 15/15 [00:00<00:00, 46.67it/s]


In [43]:
jax.device_put(pos)
jax.device_put(chg)

print("exact:")
%timeit compute_U_oneplus_exact_vectorized(pos, chg).block_until_ready()
print()

print("approx. (grid):")
%timeit compute_U_oneplus_grid(pos, chg).block_until_ready()

exact:
115 µs ± 1.71 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

approx. (grid):
3.71 ms ± 24.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Forces

In [44]:
# Double-check correctness of exact calculation by comparing different ways of doing it
f_oneplus_reference = compute_f_oneplus_exact_vectorized(pos, chg)
assert jnp.allclose(
    f_oneplus_reference, compute_f_oneplus_exact_loop(pos, chg)
)
assert jnp.allclose(
    f_oneplus_reference,
    -jax.grad(compute_U_oneplus_exact_vectorized, argnums=0)(pos, chg),
)


100%|██████████| 15/15 [00:00<00:00, 26.59it/s]


In [45]:
# Compare (semi-)manually implemented calculation of grid forces with end-to-end gradient of grid energy
_, f_oneplus_grid = compute_U_and_f_oneplus_grid(pos, chg)
assert jnp.allclose(
    f_oneplus_grid,
    -jax.grad(compute_U_oneplus_grid, argnums=0)(pos, chg),
)


In [46]:
# Compare exact and grid forces
fig, ax = plt.subplots()
ax.set_xlabel("forces $f^{1+}$ (exact)")
ax.set_ylabel("forces $f^{1+}$ (grid)")
ax.scatter(f_oneplus_reference, f_oneplus_grid)
xlim = ax.get_xlim()
ylim = ax.get_ylim()
# plot parity line and zero axes
ax.plot(xlim, xlim, color="black", zorder=-1)
ax.axhline(color="black", linewidth=0.75)
ax.axvline(color="black", linewidth=0.75)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()

<IPython.core.display.Javascript object>

# Periodic boundaries

## Construct grids


In [47]:
grids_all_levels = set_up_grids_all_levels(
    box_lengths=[length] * ndim,
    level_one_spacings=[level_one_gridspacing] * ndim,
    pbcs=[True] * ndim,
    n_levels=max_gridlevel,
    p=p,
    J_zeroplus=J_zeroplus,
)

grid_level_one = grids_all_levels[1]
grid_level_two = grids_all_levels[2]

## Construct kernel stencils

They can be reused from the non-periodic case, as they do not depend on the boundary conditions.

## Helper functions

In [48]:
import itertools
from tqdm import tqdm

In [49]:
"""
We do a basic check of pbc functionality by calculating the energy
due to the partial kernels up to the second-highest one in two ways,
exact and using the grid approximation.
The resulting interaction kernel spans multiple unit cells,
but is still finite-range.
We are thus not testing actual long-range capabilities, only whether
the wrapping of the grid operations around the grid edges is correct.
"""

max_gridlevel_for_pbc_test = max_gridlevel - 1
cutoff_for_pbc_test = level_zero_cutoff * 2**max_gridlevel_for_pbc_test
cell_size_for_pbc_test = length

n_cells_covered_by_cutoff = (jnp.ceil(cutoff_for_pbc_test / cell_size_for_pbc_test)).astype(int)
shifts_1d = jnp.arange(-n_cells_covered_by_cutoff, n_cells_covered_by_cutoff + 1)
shifts_for_pbc_test = [jnp.array(ijk) for ijk in itertools.product(*([shifts_1d, ] * ndim)) if ijk != (0,) * ndim]

@jax.jit
def kernelsum_for_pbc_test(r):
    return jnp.sum(jnp.asarray([k(r) for k in partial_kernels[1:max_gridlevel_for_pbc_test + 1]]))

In [50]:
def replicate(positions, charges):
    positions_extended = [positions.copy()] + [positions + ijk * cell_size_for_pbc_test for ijk in shifts_for_pbc_test]
    positions_extended = jnp.concatenate(positions_extended)
    charges_extended = jnp.tile(charges, len(shifts_for_pbc_test) + 1)
    
    return positions_extended, charges_extended

def periodic_compute_U_oneplus_exact_loop(positions, charges):
    positions_extended, charges_extended = replicate(positions, charges)

    U_oneplus_direct = 0.
    for i in tqdm(range(len(positions))):
        for j in range(0, len(positions_extended)):
            r_ij = jnp.linalg.norm(positions[i] - positions_extended[j])
            U_oneplus_direct += charges[i] * charges_extended[
                j] * kernelsum_for_pbc_test(r_ij)
    U_oneplus_direct *= 0.5

    return U_oneplus_direct

@jax.jit
def periodic_compute_U_oneplus_exact_vectorized(positions, charges):
    positions_extended, charges_extended = replicate(positions, charges)
    
    r_ij_vectors = positions[:, jnp.newaxis, :] - positions_extended
    # To avoid NaN gradients, we need to use a custom
    # square root function here, since the distances can be zero 
    r_ij = _sqrt((r_ij_vectors * r_ij_vectors).sum(axis=2))
    K_ij = jax.vmap(jax.vmap(kernelsum_for_pbc_test))(r_ij)
    qi_qj = charges[:, jnp.newaxis] * charges_extended
    
    return 0.5 * (qi_qj * K_ij).sum()

@jax.jit
def periodic_compute_U_oneplus_grid(positions, charges):
    return create_compute_U_oneplus(
        grids=grids_all_levels[:max_gridlevel_for_pbc_test+1],
        kernel_stencils=kernelstencils_old_symmetric[:max_gridlevel_for_pbc_test+1],
    )(positions, charges)

In [51]:
k_prime_for_pbc_test = jax.jit(jax.grad(kernelsum_for_pbc_test))


def periodic_compute_f_oneplus_exact_loop(positions, charges):
    n_dim = positions.shape[1]
    f_oneplus_direct = []
    for i in range(len(positions)):
        force = jnp.zeros(n_dim)
        for j in range(len(positions)):
            if i == j:
                continue
            r_ij_vector = positions[i] - positions[j]
            r_ij = jnp.linalg.norm(r_ij_vector)
            force -= charges[j] * (r_ij_vector / r_ij) * k_prime_for_pbc_test(r_ij)
        force *= charges[i]
        f_oneplus_direct.append(force)

    return jnp.array(f_oneplus_direct)


@jax.jit
def periodic_compute_f_oneplus_exact_vectorized(positions, charges):
    return -jax.grad(compute_U_oneplus_exact_vectorized, argnums=0)(
        positions, charges
    )


@jax.jit
def periodic_compute_U_and_f_oneplus_grid(positions, charges):
    return create_compute_U_and_f_oneplus(
        grids=grids_all_levels[:max_gridlevel_for_pbc_test+1],
        kernel_stencils=kernelstencils_old_symmetric[:max_gridlevel_for_pbc_test+1],
    )(positions, charges)

## Test end-to-end

### Energy

In [52]:
U_oneplus_periodic_reference = periodic_compute_U_oneplus_exact_vectorized(
    pos, chg
)
U_oneplus_periodic_grid = periodic_compute_U_oneplus_grid(pos, chg)
print("exact:         ", U_oneplus_periodic_reference)
print("approx. (grid):", U_oneplus_periodic_grid)

# Double-check correctness of exact reference calculation by comparing different ways of doing it
assert jnp.allclose(
    U_oneplus_periodic_reference,
    periodic_compute_U_oneplus_exact_loop(pos, chg),
)


2024-03-09 19:41:16.670821: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 1s:

  %reduce.2684 = pred[512,3375]{1,0} reduce(pred[512,3375,3]{2,1,0} %broadcast.53, pred[] %constant.51), dimensions={2}, to_apply=%region_46.2680, metadata={op_name="jit(periodic_compute_U_oneplus_grid)/jit(main)/reduce_or[axes=(2,)]" source_file="/home/florian/PhD/work/code/msm_jax_implementation/msmjax/src/msmjax/gridops_multidim.py" source_line=246}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2024-03-09 19:41:16.929096: E external/xla/xla/service/slow_operation_alarm.cc:133] The operation took 1.2583426

exact:          4.354176393685479
approx. (grid): 4.2950527582744025


 27%|██▋       | 4/15 [00:36<01:41,  9.21s/it]


KeyboardInterrupt: 

In [53]:
jax.device_put(pos)
jax.device_put(chg)

print("exact:")
%timeit periodic_compute_U_oneplus_exact_vectorized(pos, chg).block_until_ready()
print()

print("approx. (grid):")
%timeit periodic_compute_U_oneplus_grid(pos, chg).block_until_ready()

exact:
252 µs ± 10.6 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

approx. (grid):
572 µs ± 52.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Forces

In [54]:
f_oneplus_periodic_reference = -jax.grad(periodic_compute_U_oneplus_exact_vectorized, argnums=0)(pos, chg)
_, f_oneplus_periodic_grid = periodic_compute_U_and_f_oneplus_grid(pos, chg)

In [56]:
# Compare exact and grid forces
fig, ax = plt.subplots()
ax.set_xlabel("forces $f^{1+}$ (exact)")
ax.set_ylabel("forces $f^{1+}$ (grid)")
ax.scatter(f_oneplus_periodic_reference, f_oneplus_periodic_grid)
xlim = ax.get_xlim()
ylim = ax.get_ylim()
# plot parity line and zero axes
ax.plot(xlim, xlim, color="black", zorder=-1)
ax.axhline(color="black", linewidth=0.75)
ax.axvline(color="black", linewidth=0.75)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()

<IPython.core.display.Javascript object>

# Convolution experiment

In [133]:
arrshape = (3, 3)

arr = (jnp.arange(1, onp.prod(arrshape) + 1)**2).reshape(arrshape)
arr

Array([[ 1,  4,  9],
       [16, 25, 36],
       [49, 64, 81]], dtype=int64)

In [134]:
rmi = jnp.ravel_multi_index((1, 2), arr.shape)
print(rmi)
print(arr.ravel()[rmi])

5
36


In [135]:
inds_pointwise = [(0, 1), (0, 2), (1, 2)]

inds_array = jnp.array(inds_pointwise).T
inds_x = inds_array[0]
inds_y = inds_array[1]

In [136]:
rmi = jnp.ravel_multi_index((inds_x, inds_y), arr.shape)
print(rmi)
print(arr.ravel()[rmi])

[1 2 5]
[ 4  9 36]


In [137]:
mg = jnp.meshgrid(jnp.arange(arr.shape[0]), jnp.arange(arr.shape[1]), indexing="ij")
inds_all_x = mg[0].ravel()
inds_all_y = mg[1].ravel()

In [138]:
rmi = jnp.ravel_multi_index((inds_all_x, inds_all_y), arr.shape)
print(rmi)
print(arr.ravel()[rmi])

[0 1 2 3 4 5 6 7 8]
[ 1  4  9 16 25 36 49 64 81]


In [179]:
krnl_stncl = jnp.array([[0.25, 0.5, 0.25], [0.5, 1., 0.5], [0.25, 0.5, 0.25]])
krnl_stncl /= krnl_stncl.sum()

In [180]:
def wrap_idx(idx, n_pts):
    return idx % n_pts

def invalidate_idx(idx, fill_value):
    return jnp.where(idx >= 0, idx, fill_value)

In [181]:
invalidate_idx(jnp.arange(-5, 5), fill_value=arr.shape[0])

Array([3, 3, 3, 3, 3, 0, 1, 2, 3, 4], dtype=int64)

In [234]:
# # periodic
# boundary_cond_fns = [
#     functools.partial(wrap_idx, n_pts=arrshape[0]),
#     functools.partial(wrap_idx, n_pts=arrshape[1]),
# ]

# non-periodic
boundary_cond_fns = [
    functools.partial(invalidate_idx, fill_value=arrshape[0]),
    functools.partial(invalidate_idx, fill_value=arrshape[1]),
]

offsets_from_central_idx = [jnp.arange(-ks, ks + 1) for ks in kernel_sizes]

def get_neighbor_inds_in_kernelrange(*i):
    neigbor_inds_individual_axes = [
        mi + offsets for mi, offsets in zip(i, offsets_from_central_idx)
    ]
    neigbor_inds_individual_axes = [
        apply_bcs(nghbr_inds) for apply_bcs, nghbr_inds in zip(boundary_cond_fns, neigbor_inds_individual_axes)
    ]
    meshgrid = jnp.meshgrid(*neigbor_inds_individual_axes, indexing="ij")
    neighbor_multi_inds = tuple(inds.ravel() for inds in meshgrid)
    
    return neighbor_multi_inds
    
def calculate_one_element(array, *i):
    neighbor_multi_inds = get_neighbor_inds_in_kernelrange(*i)
    is_in_bounds = jnp.array([inds < s for inds, s in zip(neighbor_multi_inds, array.shape)]).all(axis=0)
    return jnp.where(is_in_bounds, array[neighbor_multi_inds] * stencil.ravel(), 0.0).sum()

In [234]:
# # periodic
# boundary_cond_fns = [
#     functools.partial(wrap_idx, n_pts=arrshape[0]),
#     functools.partial(wrap_idx, n_pts=arrshape[1]),
# ]

# # non-periodic
# boundary_cond_fns = [
#     functools.partial(invalidate_idx, fill_value=arrshape[0]),
#     functools.partial(invalidate_idx, fill_value=arrshape[1]),
# ]

# def create_custom_interaction_operator_2(
#     grid: BSplineInterpolationGrid, kernel_stencil: npt.ArrayLike
# ):
#     kernel_stencil = jnp.asarray(kernel_stencil)
#     kernelranges_individual_axes = [
#         jnp.arange(-(s // 2), (s // 2) + 1) for s in kernel_stencil.shape
#     ]
#     
#     inds_all = jnp.meshgrid(*[jnp.arange(s) for s in grid.shape], indexing="ij")
#     inds_all_flat = [i.ravel() for i in inds_all]
# 
#     def get_neighbor_inds_in_kernelrange(*ii):
#         neigbor_inds_individual_axes = [
#             i + offsets for i, offsets in zip(ii, kernelranges_individual_axes)
#         ]
#         neigbor_inds_individual_axes = [
#             ga.wrap_or_invalidate_indices(nghbr_inds) for ga, nghbr_inds in zip(grid.axes, neigbor_inds_individual_axes)
#         ]
#         meshgrid = jnp.meshgrid(*neigbor_inds_individual_axes, indexing="ij")
#         neighbor_multi_inds = tuple(inds.ravel() for inds in meshgrid)
#         
#         return neighbor_multi_inds
#         
#     def calculate_one_element(in_array, *ii):
#         neighbor_multi_inds = get_neighbor_inds_in_kernelrange(*ii)
#         is_in_bounds = jnp.array([inds < s for inds, s in zip(neighbor_multi_inds, in_array.shape)]).all(axis=0)
#         return jnp.where(is_in_bounds, in_array[neighbor_multi_inds] * kernel_stencil.ravel(), 0.0).sum()
#     
#     def apply_interaction(in_array):
#         return jax.vmap(calculate_one_element, in_axes=(None,) + (0,) * grid.ndim)(in_array, *inds_all_flat)
#     
#     return apply_interaction

In [235]:
# neighbor_multi_inds = get_neighbor_inds_in_kernelrange(0, 0)
# is_in_bounds = jnp.array([inds < s for inds, s in zip(neighbor_multi_inds, arrshape)]).all(axis=0)
# jnp.where(is_in_bounds, arr[neighbor_multi_inds] * krnl_stncl.ravel(), 0.0).sum()

In [236]:
calculate_one_element(0, 0)

Array(4.3125, dtype=float64)

In [224]:
inds_all = jnp.meshgrid(*[jnp.arange(s) for s in arrshape], indexing="ij")

In [225]:
inds_all_flat = [i.ravel() for i in inds_all]

In [232]:
jax.vmap(jax.vmap(calculate_one_element))(*inds_all)

Array([[ 4.3125,  8.625 ,  8.8125],
       [17.625 , 30.    , 27.625 ],
       [23.8125, 38.625 , 34.3125]], dtype=float64)

In [233]:
jax.vmap(calculate_one_element)(*inds_all_flat).reshape(arrshape)

Array([[ 4.3125,  8.625 ,  8.8125],
       [17.625 , 30.    , 27.625 ],
       [23.8125, 38.625 , 34.3125]], dtype=float64)

In [201]:
jax.scipy.signal.convolve(arr, krnl_stncl, mode="same", method="direct")

Array([[ 4.3125,  8.625 ,  8.8125],
       [17.625 , 30.    , 27.625 ],
       [23.8125, 38.625 , 34.3125]], dtype=float64)